# NB01 — MAG feature matrix

**Goal:** Build the per-MAG feature matrix linking metal-gene density to environmental targets.

**Steps:**
1. Pull soil bacterial MAG metadata from Spark (`mag_coordinates` + `genome_metadata`; ENVO soil/rhizosphere filter via `sample_microntology`). Cached to `data/mag_metadata_full_cache.parquet`. Spark is stopped after caching to free JVM memory.
2. Load pre-computed eggnog KO annotations from `data/mag_ko_annotations/` (Hive-partitioned by `sample_id`; built by `scripts/process_eggnog_to_parquet.py`). PyArrow reads only the partitions matching metadata sample_ids — avoids scanning all 169 GB.
3. Count distinct primary KOs and subcategory KOs per MAG.
4. Filter MAGs: completeness ≥70%, contamination ≤10%, Bacteria.
5. Compute `ko_per_mb_*` densities (KO count / genome size in Mb).
6. Spatial join to CSU metal mobility grid (≤50 km).
7. Save `data/mag_feature_matrix.parquet`.

**ENVO filter:** Only samples with `environment_term LIKE '%soil%' OR '%rhizosphere%'` are included.

**Output:** `data/mag_feature_matrix.parquet`, `data/nb01_build_summary.json`.

In [1]:
print("NB01 executing — building MAG feature matrix.")

NB01 executing — building MAG feature matrix.


In [2]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from mag_utils import filter_mag_metadata, get_primary_ko_set, get_subcategory_ko_sets
from env_utils import batch_csu_join, CSU_TARGETS
from spire_api import SPIREClient

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

# Spark session — try berdl_notebook_utils (on-cluster), then repo script, then None
try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
    print("Spark connected via berdl_notebook_utils:", spark.version)
except Exception:
    try:
        repo_scripts = str(Path.cwd().parents[1] / 'scripts')
        sys.path.insert(0, repo_scripts)
        from get_spark_session import get_spark_session
        spark = get_spark_session()
        print("Spark connected via repo scripts:", spark.version)
    except Exception as e2:
        spark = None
        print(f"No Spark session ({e2}) — Spark metadata query will be skipped.")

DATA_DIR = Path.cwd().parent / 'data'
with open(DATA_DIR / 'spire_probe_results.json') as f:
    probe = json.load(f)

print(f"Use SPIRE downloads: {probe.get('use_spire_downloads', False)}")

INFO HTTP Request: GET http://mms.prod:8000/workspaces/me/sql-warehouse-prefix "HTTP/1.1 200 OK"


Spark connected via berdl_notebook_utils: 4.0.1
Use SPIRE downloads: True


In [3]:
# --------------------------------------------------------------------------
# Pull soil bacterial MAG metadata from Spark (or load from cache).
# Positive ENVO filter: sample must have at least one 'soil' or 'rhizosphere'
# environment term in sample_microntology.
# Soil features come from sample_environment (LEFT JOIN).
# --------------------------------------------------------------------------
_META_CACHE = DATA_DIR / 'mag_metadata_full_cache.parquet'

if spark is not None:
    mag_meta_df = spark.sql("""
        SELECT DISTINCT
            mc.mag_id,
            mc.sample_id,
            mc.latitude,
            mc.longitude,
            gm.genome_size         AS genome_size_bp,
            gm.completeness,
            gm.contamination,
            gm.domain,
            gm.genus,
            se.sg_phh2o_0_5cm_mean AS ph_h2o,
            se.sg_clay_0_5cm_mean  AS clay_content,
            se.sg_soc_0_5cm_mean   AS organic_carbon_density
        FROM refdata.spire.mag_coordinates mc
        JOIN refdata.spire.genome_metadata  gm ON mc.mag_id = gm.genome_id
        LEFT JOIN refdata.spire.sample_environment se ON mc.sample_id = se.sample_id
        WHERE mc.latitude IS NOT NULL
          AND mc.longitude IS NOT NULL
          AND gm.domain = 'Bacteria'
          AND EXISTS (
            SELECT 1 FROM refdata.spire.sample_microntology sm
            WHERE sm.sample_id = mc.sample_id
              AND (sm.environment_term LIKE '%soil%'
                OR sm.environment_term LIKE '%rhizosphere%')
          )
          AND NOT EXISTS (
            SELECT 1 FROM refdata.spire.sample_microntology sm
            WHERE sm.sample_id = mc.sample_id
              AND (sm.environment_term LIKE '%host%'
                OR sm.environment_term LIKE '%gut%'
                OR sm.environment_term LIKE '%clinical%')
          )
    """).toPandas()
    mag_meta_df.attrs = {}  # Spark attaches non-serializable PlanMetrics to attrs
    mag_meta_df.to_parquet(_META_CACHE, index=False)
    print(f"Saved metadata cache: {_META_CACHE}")
    # Stop Spark now — JVM holds ~8-10 GB; freeing it gives DuckDB/PyArrow headroom
    spark.stop()
    spark = None
    print("Spark session stopped.")
elif _META_CACHE.exists():
    mag_meta_df = pd.read_parquet(_META_CACHE)
    print(f"Loaded metadata from cache (no Spark): {_META_CACHE}")
else:
    raise RuntimeError(
        "Spark is unavailable and no metadata cache exists. "
        "Run once with Spark to build data/mag_metadata_full_cache.parquet."
    )

print(f"Soil bacterial MAGs (ENVO soil/rhizosphere filter): {len(mag_meta_df):,}")
print(f"Unique samples: {mag_meta_df['sample_id'].nunique():,}")
print(mag_meta_df[['mag_id', 'sample_id', 'completeness', 'genome_size_bp']].head(3))

Saved metadata cache: /home/hmacgregor/BERIL-research-observatory/projects/metagenomic_environment_prediction/data/mag_metadata_full_cache.parquet
Spark session stopped.
Soil bacterial MAGs (ENVO soil/rhizosphere filter): 22,312
Unique samples: 2,262
               mag_id     sample_id  completeness  genome_size_bp
0  spire_mag_00059750  SAMN06266398         65.37       3568022.0
1  spire_mag_00078987  SAMN06239022         78.62       2929778.0
2  spire_mag_00096554  SAMN06343853         98.08       9657842.0


In [4]:
# --------------------------------------------------------------------------
# Load pre-computed eggnog annotations and compute per-MAG KO counts.
#
# Reads from data/mag_ko_annotations/ (Hive-partitioned by sample_id).
# Only reads partitions for sample_ids present in mag_meta_df — avoids
# scanning all 2,251 partitions (169 GB) and stays within cgroup limit.
# Uses pq.ParquetFile to read each file directly (bypasses dataset API
# to avoid the sample_id large_string/dictionary type merge conflict).
# --------------------------------------------------------------------------
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc
import re

primary_kos = get_primary_ko_set()
subcat_kos  = get_subcategory_ko_sets()
all_curated = list(primary_kos | set().union(*subcat_kos.values()))  # 730 KOs
curated_arr  = pa.array(all_curated, type=pa.utf8())

annot_dir = DATA_DIR / 'mag_ko_annotations'
if not annot_dir.exists() or not any(annot_dir.glob('**/*.parquet')):
    raise RuntimeError(
        f"{annot_dir} not found or empty. "
        "Run: python scripts/process_eggnog_to_parquet.py --merge-only"
    )

sample_ids_needed = set(mag_meta_df['sample_id'].unique())
print(f"Reading annotations for up to {len(sample_ids_needed):,} sample partitions...", flush=True)

chunks = []
n_found = 0
for sid in sorted(sample_ids_needed):
    part_path = annot_dir / f'sample_id={sid}' / 'data.parquet'
    if not part_path.exists():
        continue
    n_found += 1
    # ParquetFile.read() bypasses dataset discovery — no Hive schema conflict
    tbl = pq.ParquetFile(str(part_path)).read(columns=['mag_id', 'ko_id'])
    # Cast ko_id to utf8 in case it was written as large_utf8
    ko_col = tbl.column('ko_id')
    if ko_col.type != pa.utf8():
        ko_col = ko_col.cast(pa.utf8())
    mask = pc.is_in(ko_col, value_set=curated_arr)
    tbl = tbl.filter(mask)
    if len(tbl):
        chunks.append(tbl.to_pandas())
    del tbl

annot_df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=['mag_id', 'ko_id'])
del chunks
print(f"Partitions found: {n_found:,}  |  Curated (MAG, KO) pairs: {len(annot_df):,}  |  MAGs: {annot_df['mag_id'].nunique():,}")

# Count distinct KOs per MAG per category
mag_ko = annot_df.groupby('mag_id')['ko_id'].apply(frozenset)
del annot_df

subcat_rename = {
    'n_ko_resistance_detoxification':  'n_ko_resistance',
    'n_ko_transport_homeostasis':      'n_ko_transport',
    'n_ko_sensing_regulation':         'n_ko_sensing',
    'n_ko_metal-dependent_metabolism': 'n_ko_metabolism',
    'n_ko_cofactor_biosynthesis':      'n_ko_cofactor',
    'n_ko_unknown':                    'n_ko_unknown',
}

records = []
for mag_id, ko_set in mag_ko.items():
    row = {'mag_id': mag_id, 'n_ko_primary': len(ko_set & primary_kos)}
    for cat, cat_kos in subcat_kos.items():
        col = 'n_ko_' + cat.lower().replace(' ', '_').replace('/', '_')
        row[col] = len(ko_set & cat_kos)
    records.append(row)

ko_counts_df = pd.DataFrame(records)
ko_counts_df = ko_counts_df.rename(columns={k: v for k, v in subcat_rename.items()
                                             if k in ko_counts_df.columns})

print(f"KO counts computed: {len(ko_counts_df):,} MAGs")
print(ko_counts_df[['n_ko_primary', 'n_ko_resistance', 'n_ko_transport']].describe().round(1))

# Drop non-standard sample IDs
_STANDARD = re.compile(r'^(SAMN|SAMEA|SRS|ERS|DRS)\d+$')
standard_mask = mag_meta_df['sample_id'].str.match(_STANDARD)
n_dropped = (~standard_mask).sum()
if n_dropped:
    print(f"Dropping {n_dropped:,} MAGs with non-standard sample IDs")
run_df = mag_meta_df[standard_mask].copy()

mag_density_df = run_df.merge(ko_counts_df, on='mag_id', how='inner')
print(f"After metadata merge: {len(mag_density_df):,} MAGs")

Reading annotations for up to 2,262 sample partitions...


Partitions found: 2,251  |  Curated (MAG, KO) pairs: 117,356,222  |  MAGs: 21,611


KO counts computed: 21,611 MAGs
       n_ko_primary  n_ko_resistance  n_ko_transport
count       21611.0          21611.0         21611.0
mean          157.7             19.5            43.7
std            35.8              4.0            14.6
min             3.0              0.0             0.0
25%           144.0             18.0            35.0
50%           170.0             21.0            47.0
75%           183.0             22.0            55.0
max           204.0             23.0            70.0
Dropping 24 MAGs with non-standard sample IDs
After metadata merge: 21,611 MAGs


In [5]:
# --------------------------------------------------------------------------
# Quality filter: completeness ≥70%, contamination ≤10%, Bacteria
# (Soil environment filter already applied in Spark SQL EXISTS clause)
# --------------------------------------------------------------------------
genome_meta = mag_density_df.copy()

# filter_mag_metadata expects 'kingdom' column
genome_meta = genome_meta.rename(columns={'domain': 'kingdom'})
genome_meta = filter_mag_metadata(genome_meta)
genome_meta = genome_meta.rename(columns={'kingdom': 'domain'})

print(f"Bacterial MAGs passing QC (completeness ≥70%, contamination ≤10%): {len(genome_meta):,}")
print(genome_meta[['completeness', 'contamination']].describe().T[['min', 'mean', 'max']].round(2))

Bacterial MAGs passing QC (completeness ≥70%, contamination ≤10%): 15,957
                min   mean    max
completeness   70.0  87.99  100.0
contamination   0.0   3.37   10.0


In [6]:
# --------------------------------------------------------------------------
# Compute per-Mb densities from downloaded KO counts
# --------------------------------------------------------------------------
def n_ko_to_density(n_ko: int, genome_size_bp: float) -> float:
    if genome_size_bp is None or genome_size_bp <= 0:
        return float('nan')
    return int(n_ko) / (float(genome_size_bp) / 1e6)

density_df = genome_meta.copy()

SUBCAT_MAP = {
    'ko_per_mb_primary':    'n_ko_primary',
    'ko_per_mb_resistance': 'n_ko_resistance',
    'ko_per_mb_transport':  'n_ko_transport',
    'ko_per_mb_sensing':    'n_ko_sensing',
    'ko_per_mb_metabolism': 'n_ko_metabolism',
    'ko_per_mb_cofactor':   'n_ko_cofactor',
}

for density_col, count_col in SUBCAT_MAP.items():
    if count_col not in density_df.columns:
        print(f"Warning: {count_col} not in columns — skipping {density_col}")
        density_df[density_col] = float('nan')
        continue
    density_df[density_col] = density_df.apply(
        lambda r, cc=count_col: n_ko_to_density(r[cc], r['genome_size_bp']), axis=1
    )

print(f"Density rows: {len(density_df):,}")
print(density_df[['mag_id', 'ko_per_mb_primary', 'ko_per_mb_resistance',
                   'ko_per_mb_transport']].describe().round(4))

Density rows: 15,957
       ko_per_mb_primary  ko_per_mb_resistance  ko_per_mb_transport
count         15957.0000            15957.0000           15957.0000
mean             51.6006                6.4574              13.8411
std              40.2179                5.1028              10.8671
min               1.0310                0.0000               0.0000
25%              30.9156                3.8848               7.8307
50%              42.7606                5.3318              11.6640
75%              58.6657                7.2359              16.5945
max             504.8633               61.2840             151.7508


In [7]:
# --------------------------------------------------------------------------
# Add CSU metal mobility targets via spatial join (PF1 fractions)
# Soil features (ph_h2o, clay_content, organic_carbon_density) are already
# in density_df from the Spark SQL query via sample_environment.
# --------------------------------------------------------------------------
feature_df = density_df.copy()

# CSU metal mobility spatial join
# Grid lives in microbeatlas_metal_ecology project; rename lat/lon columns first
PROJECTS_DIR = Path.cwd().parents[1]
csu_grid_path = PROJECTS_DIR / 'microbeatlas_metal_ecology' / 'data' / 'csu_metal_mobility_grid.parquet'

if csu_grid_path.exists():
    csu_grid = pd.read_parquet(csu_grid_path)
    csu_grid = csu_grid.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
    feature_df = batch_csu_join(feature_df, csu_grid)
    print(f"After CSU join: {feature_df['PF1_Cu'].notna().sum()} MAGs with CSU data")
else:
    print(f"WARNING: CSU grid not found at {csu_grid_path}")
    for col in ['PF1_As', 'PF1_Cd', 'PF1_Cr', 'PF1_Cu', 'PF1_Hg', 'PF1_Pb']:
        feature_df[col] = float('nan')

print(f"ph_h2o non-null:           {feature_df['ph_h2o'].notna().sum()}")
print(f"clay_content non-null:     {feature_df['clay_content'].notna().sum()}")
print(f"organic_carbon_density nn: {feature_df['organic_carbon_density'].notna().sum()}")

After CSU join: 15368 MAGs with CSU data
ph_h2o non-null:           13182
clay_content non-null:     13182
organic_carbon_density nn: 13182


In [8]:
# --------------------------------------------------------------------------
# Save output
# --------------------------------------------------------------------------
out_path = DATA_DIR / 'mag_feature_matrix.parquet'
feature_df.to_parquet(out_path, index=False)
print(f"Saved: {out_path}  ({len(feature_df):,} MAGs)")

summary = {
    'source': 'spire_download_endpoints',
    'n_mags_total': len(feature_df),
    'n_with_csu': int(feature_df['PF1_Cu'].notna().sum()),
    'n_with_soilgrids': int(feature_df['ph_h2o'].notna().sum()),
    'mean_ko_per_mb_primary': float(feature_df['ko_per_mb_primary'].mean()),
}
with open(DATA_DIR / 'nb01_build_summary.json', 'w') as f:
    import json
    json.dump(summary, f, indent=2)

print(summary)

Saved: /home/hmacgregor/BERIL-research-observatory/projects/metagenomic_environment_prediction/data/mag_feature_matrix.parquet  (15,957 MAGs)
{'source': 'spire_download_endpoints', 'n_mags_total': 15957, 'n_with_csu': 15368, 'n_with_soilgrids': 13182, 'mean_ko_per_mb_primary': 51.600571070241}
